## `LimitRange` and `ResourceQuota` — namespace governance

Once teams share a cluster, you don't want a `BestEffort` pod hogging a node, or a stray test requesting 100 cores. Two admin objects govern that.

**`LimitRange`** (namespace-scoped) does two things — **fill in defaults** when a pod omits requests/limits, and **reject** values outside an allowed range:

```yaml
kind: LimitRange
metadata: { name: team-a-defaults, namespace: team-a }
spec:
  limits:
    - type: Container
      default:        { cpu: 500m, memory: 256Mi }   # when limits unset
      defaultRequest: { cpu: 100m, memory: 64Mi }    # when requests unset
      max:            { cpu: "4",  memory: 4Gi }     # reject above
      min:            { cpu: 10m,  memory: 16Mi }    # reject below
```

**`ResourceQuota`** (namespace-scoped) caps the **sum** across the whole namespace — requests, limits, and object counts:

```yaml
kind: ResourceQuota
metadata: { name: team-a-budget, namespace: team-a }
spec:
  hard:
    requests.cpu: "10"
    limits.memory: 40Gi
    pods: "50"
    persistentvolumeclaims: "10"
```

Hit 10 cores of requests and the next pod that would push over is **rejected at creation**; `kubectl describe quota -n team-a` shows used vs hard.

The CKA wants the distinction crisp: **`LimitRange` shapes individual objects (defaults + per-object caps); `ResourceQuota` caps the namespace total.** A subtle interaction — with a `ResourceQuota` on requests/limits, every pod *must* set them, so pair it with a `LimitRange` that supplies defaults. On our map both live in the **Scheduling** box as the multi-tenant guardrails around requests and limits.